In [1]:
import pandas as pd
from scipy.stats import shapiro

In [2]:
df = pd.read_csv(r"C:\Users\HP\Desktop\Projects - Data Analysis\RSF\RSF\ckm_staged_ehr_data.csv")
df

,patient_id,year_of_birth,death_date,encounter_id,encounter_datetime,discharge_datetime,height_value,weight_value,bmi_value,waist_circumference_value,...,nationality_Bangladesh,nationality_Egypt,nationality_India,nationality_Jordan,nationality_Other,nationality_Pakistan,nationality_Philippines,nationality_UAE,nationality_United Kingdom,ckm_stage
0,PT000001,1970,NaN,ENC00000001,2015,2015,169.7,93.6,32.5,90.2,...,0,0,0,0,0,1,0,0,0,1
1,PT000002,1970,NaN,ENC00000036,2015,2015,NaN,NaN,NaN,NaN,...,1,0,0,0,0,0,0,0,0,0
2,PT000003,1970,NaN,ENC00000079,2015,2015,189.7,73.1,20.3,85.5,...,0,0,0,0,0,0,1,0,0,0
3,PT000004,1970,NaN,ENC00000115,2015,2015,165.0,83.9,30.8,86.8,...,0,0,1,0,0,0,0,0,0,1
4,PT000005,1970,NaN,ENC00000158,2015,2015,161.1,101.8,39.2,84.2,...,0,0,0,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,PT001196,1970,2022.0,ENC00028566,2015,2015,171.2,52.8,18.0,76.3,...,0,0,1,0,0,0,0,0,0,0
1196,PT001197,1970,NaN,ENC00028584,2015,2015,172.3,87.4,29.4,73.5,...,0,0,0,0,0,1,0,0,0,1
1197,PT001198,1970,NaN,ENC00028610,2015,2015,152.0,57.1,24.7,77.7,...,0,0,0,0,0,0,1,0,0,0
1198,PT001199,1970,2018.0,ENC00028624,2015,2015,164.4,81.5,30.2,108.2,...,0,0,0,0,0,0,1,0,0,1


In [4]:
df.columns

Index(['patient_id', 'year_of_birth', 'death_date', 'encounter_id',
       'encounter_datetime', 'discharge_datetime', 'height_value',
       'weight_value', 'bmi_value', 'waist_circumference_value', 'systolic_bp',
       'diastolic_bp', 'heart_rate', 'ALT', 'Fasting Glucose',
       'HDL Cholesterol', 'HbA1c', 'Hemoglobin', 'LDL Cholesterol',
       'Potassium', 'Serum Albumin', 'Serum Creatinine', 'Sodium',
       'Triglycerides', 'UACR', 'Uric Acid', 'eGFR',
       'Acute myocardial infarction, unspecified',
       'Atherosclerotic heart disease, subclinical',
       'Cerebral infarction, unspecified', 'Chronic kidney disease, stage 3',
       'Chronic kidney disease, stage 4', 'Chronic kidney disease, stage 5',
       'Essential (primary) hypertension', 'Heart failure, unspecified',
       'Hyperlipidemia, unspecified', 'Obesity, unspecified',
       'Peripheral artery disease, unspecified extremity', 'Prediabetes',
       'Proteinuria, severely increased',
       'Type 2 diabetes 

In [5]:
biomarker_cols = ['eGFR', 'UACR', 'HbA1c', 'bmi_value', 'LDL Cholesterol', 'systolic_bp']
for col in biomarker_cols:
    distribution, pvalue = shapiro(x=df[col], nan_policy='omit')
    alpha = 0.05
    if pvalue <= alpha:
        print(f"{col} has skewed distribution of data.")
    elif pvalue > alpha:
        print(f"{col} has normally distributed data.")
    else:
        print(f"Error: {pvalue}")

eGFR has skewed distribution of data.
UACR has skewed distribution of data.
HbA1c has skewed distribution of data.
bmi_value has skewed distribution of data.
LDL Cholesterol has normally distributed data.
systolic_bp has skewed distribution of data.


In [ ]:
egfr_desc_stats = df.groupby('ckm_stage')['egfr'].describe().reset_index()
egfr_desc_stats

In [ ]:
uacr_desc_stats = df.groupby('ckm_stage')['uacr'].describe().reset_index()
uacr_desc_stats

In [ ]:
hba1c_desc_stats = df.groupby('ckm_stage')['hba1c'].describe().reset_index()
hba1c_desc_stats

In [ ]:
ldl_desc_stats = df.groupby('ckm_stage')['ldl'].describe().reset_index()
ldl_desc_stats

In [ ]:
bmi_desc_stats = df.groupby('ckm_stage')['bmi'].describe().reset_index()
bmi_desc_stats

In [ ]:
sbp_desc_stats = df.groupby('ckm_stage')['sbp'].describe().reset_index()
sbp_desc_stats

ANOVA (Normal Distribution)

In [6]:
from scipy.stats import f_oneway
def quick_anova(df, cat_var, quant_var):
    var_dir = {0:[], 1:[], 2:[], 3:[], 4:[]}
    for i in range(5):
        var_dir[i] = df.loc[(df[cat_var] == i), quant_var].dropna()
    f_statistic, p_value = f_oneway(var_dir[0], var_dir[1], var_dir[2], 
                                    var_dir[3], var_dir[4])
    return p_value

In [8]:
ldl_anova_pvalue = quick_anova(df, 'ckm_stage', 'LDL Cholesterol')
print(f"ANOVA results: \nldl: {ldl_anova_pvalue}")

ANOVA results: 
ldl: nan


C:\Users\HP\AppData\Local\Temp\ipykernel_3800\913644633.py:6: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  f_statistic, p_value = f_oneway(var_dir[0], var_dir[1], var_dir[2],


Kruskal-Wallis (Skewed Distribution)

In [ ]:
from scipy.stats import kruskal
def quick_kruskal(df, cat_var, quant_var):
    var_dir = {0:[], 1:[], 2:[], 3:[], 4:[]}
    for i in range(5):
        var_dir[i] = df.loc[df[cat_var] == i, quant_var].dropna()
    result = kruskal(var_dir[0], var_dir[1], var_dir[2], 
                                    var_dir[3], var_dir[4])
    return result

In [ ]:
egfr_kruskal_result = quick_kruskal(df, 'ckm_stage', 'egfr')
uacr_kruskal_result = quick_kruskal(df, 'ckm_stage', 'uacr')
hba1c_kruskal_result = quick_kruskal(df, 'ckm_stage', 'hba1c')
bmi_kruskal_result = quick_kruskal(df, 'ckm_stage', 'bmi')
sbp_kruskal_result = quick_kruskal(df, 'ckm_stage', 'sbp')
print(f"Kruskal-Wallis results: \nuacr: {uacr_kruskal_result.pvalue}" + 
      f"\nhba1c: {hba1c_kruskal_result.pvalue} \negfr : {egfr_kruskal_result.pvalue}" +
      f"\nbmi: {bmi_kruskal_result.pvalue} \nsbp: {sbp_kruskal_result.pvalue}")